Adapted from original script by David Mann.

# Setup

In [1]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
from typing import Counter

In [66]:
INCIDENCE_ANGLE = 62.5*np.pi / 180  # 62.5 deg
N_PAIRS = 9

# Functions

In [67]:
class Diodes():
    def __init__(self):
        self.diodes = []
        self.twins = []

    def add_diode(self, x, y, z):
        vector = np.array([x, y, z])
        norm_vector = vector / np.linalg.norm(vector)    
        self.diodes.append(norm_vector)
        tv = -norm_vector
        self.twins.append(tv)


In [68]:
def spherical_to_cartesian(theta, phi):
    x = np.sin(phi) * np.cos(theta)
    y = np.sin(phi) * np.sin(theta)
    z = np.cos(phi)
    return x, y, z

In [69]:
pip install imageio

Note: you may need to restart the kernel to use updated packages.


In [70]:
def optimize_antipodal_pairs(n_pairs, iterations=1000, learning_rate=0.02):
    """
    Optimizes the placement of antipodal pairs on a sphere. It is impossible
    to evenly space points on a sphere unless N = 4, 6, 8, 12, or 20. Additionally,
    the Fibonacci spiral does not guarantee antipodal points. Thus, this method
    simulates electrostatic repulsion to find a near-optimal spacing while
    adhering to the antipode constraint. 

    Parameters:
    - n_pairs: Number of antipodal pairs to distribute.
    - iterations: Number of simulation steps.
    - learning_rate: Controls the movement speed during optimization.

    Returns:
    - points: Array of shape (n_pairs, 3) with Cartesian coordinates.
    """
    # Initialize random points on the sphere
    phi = np.arccos(1 - 2 * np.random.rand(n_pairs))
    theta = 2 * np.pi * np.random.rand(n_pairs)
    x, y, z = spherical_to_cartesian(phi, theta)

    points = np.vstack((np.column_stack((x, y, z)), np.column_stack((-x, -y, -z))))

    for _ in range(iterations):
        forces = np.zeros_like(points)
        for i in range(n_pairs):
            antipode_idx = i + n_pairs
            to_compare = np.delete(points, [i, antipode_idx], axis=0)
            diffs = points[i] - to_compare
            distances = np.linalg.norm(diffs, axis=1, keepdims=True)
            
            # Calculate repulsive forces
            repulsive_forces = diffs / (distances ** 3 + 1e-9)
            net_force = np.sum(repulsive_forces, axis=0)
            forces[i] += net_force
            forces[antipode_idx] -= net_force  # Equal and opposite force on antipode

        points += learning_rate * forces
        # Normalize to keep points on the sphere
        points /= np.linalg.norm(points, axis=1, keepdims=True)

    ## Only return first half of points. In Diode, the antipodes are generated
    ## precisely so as to ensure exact opposites.
    return points[:len(points) // 2]

In [71]:
def checkCoverage(diodes: Diodes, granularity=100, plot=False):
    '''
    This method generates a meshgrid of points on the unit sphere to test diode
    coverage. It can optionally plot a 3D graph of the sphere.
    Params:
    diodes - A Diodes object with populated diodes.
    granularity - The relative fineness of the simulation. Larger numbers (ie. 300)
        will be more computationally expensive but more accurate.
    plot - True if plot is desired, false else.
    '''
    theta, phi = np.linspace(0, 2 * np.pi, granularity), np.linspace(0, np.pi, granularity)
    THETA, PHI = np.meshgrid(theta, phi)
    X, Y, Z = spherical_to_cartesian(THETA, PHI)
    points = np.array([X.flatten(), Y.flatten(), Z.flatten()]).T
    color_counts = np.zeros(points.shape[0])

    all_diodes = diodes.diodes + diodes.twins
    for diode in all_diodes:
        dot_products = np.dot(points, diode)
        color_counts[np.arccos(dot_products) < INCIDENCE_ANGLE] += 1

    # Specify the colors desired for n photodiode coverage on the unit sphere.
    COVERAGE_MAP = {
        0: 'blue',
        1: 'green',
        2: 'yellow',
        3: 'orange',
        4: 'red',
        5: 'purple',
        6: 'gray',
        7: 'black',
        8: 'black',
        9: 'black',
        10: 'black',
        11: 'black',
        12: 'black',
        13: 'black',
        14: 'black',
        15: 'black',
        16: 'black',
        17: 'black',
        18: 'black',
    }

    colors = [COVERAGE_MAP[i] for i in color_counts]
    counts = Counter(colors)

    total = 0
    for value, count in counts.items():
        total += count
        #print(f"Color: {value}, Count: {count}")
    coverage_4 = (total - counts['orange'] - counts['green'] - counts['blue']- counts['yellow']) / total
    coverage_3 = (total - counts['green'] - counts['blue']- counts['yellow']) / total
    if(plot):
        print(f'Percent coverage >=3 Diodes: {coverage_3}')
        print(f'Percent coverage >=4 Diodes: {coverage_4}')

    if plot:
        fig = plt.figure()
        ax = fig.add_subplot(1,1,1, projection='3d')
        ax.axis('off')
        plot = ax.scatter(X, Y, Z, c=colors, s=0.5)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_zticklabels([])

        custom_legend = {
            'blue': '0',
            'green': '1',
            'yellow': '2',
            'orange': '3',
            'red': '4',
            'purple': '5',
            'gray': '6',
            'black': '7+',
        }
        handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=5, label=label) for color, label in custom_legend.items()]
        ax.legend(handles=handles, title='Coverage', loc='upper right')

        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        ax.set_zlim(-1, 1)
        ax.set_box_aspect([1, 1, 1])  # Equal aspect ratio for x, y, z axes
        ax.set_title('Photodiode Coverage on the Rotation Sphere', fontweight='bold')

        plt.show()
    return coverage_4

In [72]:
def plot_points(points):
    '''
    This method plots the points representing diodes on the unit rotation sphere.
    It also draws lines between each diode and its antipode to help with 
    visualization.
    Parameters:
    points - a list of points (excluding antipodes) representing photodiode placement.
    '''
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(points[:, 0], points[:, 1], points[:, 2], color='blue', label='Points', s=1)
    ax.scatter(-points[:, 0], -points[:, 1], -points[:, 2], color='red', label='Antipodes', s=1)

    # Draw lines connecting antipodal pairs
    for i in range(len(points)):
        x_vals = [points[i, 0], -points[i, 0]]
        y_vals = [points[i, 1], -points[i, 1]]
        z_vals = [points[i, 2], -points[i, 2]]
        ax.plot(x_vals, y_vals, z_vals, color='green', linewidth=0.8)

    # Add unit sphere
    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)
    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones(np.size(u)), np.cos(v))

    ax.plot_surface(x, y, z, color='c', alpha=0.2, zorder=-1)

    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_zlim(-1, 1)
    ax.set_box_aspect([1, 1, 1])  # Equal aspect ratio for x, y, z axes
    ax.set_title('Visualization of Diode Pairs on Rotation Sphere')

    plt.show()

# Script

In [73]:
plt.close('all')

First, we define our constants

Next, run `optimize_antipodal_pairs` to estimate the optimal spacing

In [74]:
optimized_points = optimize_antipodal_pairs(N_PAIRS)

The above proved that the algorithm worked. Now, we want to find the absolutely best set of pairs

In [76]:
from pts2planes import dowork, angle_between

Step 1: Find the map that gives you the set of photodiodes where the ramp they'd be on has the smallest angles possible, while making sure the 4+ coverage is still good

In [77]:
minimum_angle_so_far = 10000
best_map = {}
best_orig_map = {}
best_optimized_pts = []
for _ in range(10000):
    optimized_points = optimize_antipodal_pairs(N_PAIRS)
    d = Diodes()
    for i, point in enumerate(optimized_points):
        d.add_diode(optimized_points[i][0], optimized_points[i][1], optimized_points[i][2])
    if(checkCoverage(d)<0.95):
        continue
    else:
        missing, mapped, minimum_angle, original_mapped = dowork(list(optimized_points))
        
        if(missing):
            continue
        else:
            if(minimum_angle < minimum_angle_so_far):
                minimum_angle_so_far = minimum_angle
                best_map = mapped
                best_orig_map = original_mapped
                best_optimized_pts = optimized_points

Take a look at your points. Do you notice a pattern?

In [90]:
plane_to_nv = {}
for key in best_orig_map.keys():
    for point in best_orig_map[key]:
        print(np.rad2deg(point[1]))

32.86782059877681
31.85521473091566
26.9554817770714
33.025113404750094
33.35813582327939
26.517172590668515
32.31576583724476
31.63985905705586
24.962341435739056


Since all the angles are relatively close to 30, let's SLERP them to 30

In [ ]:
normal_vectors = {
    "Plane 1": np.array([1, 0, 0]),
    "Plane 2": np.array([0, 1, 0]),
    "Plane 3": np.array([0, -1, 0]),
    "Plane 4": np.array([0, 0, -1]),
    "Plane 5": np.array([0 ,0, 1]),
    "Plane 6": np.array([-1, 0, 0])
}
angles = [30]
new_map = {
    "Plane 1": [],
    "Plane 2": [],
    "Plane 3": [],
    "Plane 4": [],
    "Plane 5": [],
    "Plane 6": []
}
new_vecs = []
original_vecs = []
for key in best_orig_map.keys():
    for point in best_orig_map[key]:
        original_vec = point[0]
        angle = point[1]
        closest_angle = 100000000
        for a in angles:
            if(abs(closest_angle - angle) > abs(a - angle)):
                closest_angle = a
        #uses SLERP to find the new vector on the great circle arc that is exactly 30 degrees from the plane's normal vector --> this means its ramp will be 30 degrees
        new_vector = np.sin(angle - np.deg2rad(closest_angle))/np.sin(angle) * normal_vectors[key] + np.sin(np.deg2rad(closest_angle))/np.sin(angle) * original_vec
        
        new_vecs.append(new_vector)
        original_vecs.append(original_vec)
        new_map[key].append(new_vector)

Then, add the diodes to the Diode class and run `checkCoverage` to analyze the output

In [ ]:
# Add diodes to the class
d = Diodes()
for i, point in enumerate(new_vecs):
    print(f"Point {i + 1}: {point}")
    d.add_diode(point[0], point[1], point[2])

checkCoverage(d, granularity=500, plot=True)

Convert to different formats for Structures

In [ ]:
from pts2planes import cartesian_to_spherical
from scipy.spatial.transform import Rotation
reverse_map = {
    "Plane 1": [],
    "Plane 2": [],
    "Plane 3": [],
    "Plane 4": [],
    "Plane 5": [],
    "Plane 6": []
}
for key in new_map.keys():
    for point in new_map[key]:
        if(key == 'Plane 1'):
            reverse_map['Plane 6'].append(point * -1)
        if(key == 'Plane 2'):
            reverse_map['Plane 3'].append(point * -1)
        if(key == 'Plane 3'):
            reverse_map['Plane 2'].append(point * -1)
        if(key == 'Plane 4'):
            reverse_map['Plane 5'].append(point * -1)
        if(key == 'Plane 5'):
            reverse_map['Plane 4'].append(point * -1)
        if(key == 'Plane 6'):
            reverse_map['Plane 1'].append(point * -1)
for key in new_map.keys():
    new_map[key] = reverse_map[key] + new_map[key]
for key in new_map.keys():
    print(key)
    for point in new_map[key]:
        print("Original point: ", point)
        pt = np.array(point)
        r = np.linalg.norm(pt)
        if(key in ['Plane 4', 'Plane 5']):
            phi = np.arctan2(pt[0], pt[2])
            psi = np.arccos(pt[1]/r)
            print(f"Phi {np.rad2deg(phi)}, Psi: {np.rad2deg(psi)}, Angle: {np.rad2deg(angle_between(point, normal_vectors[key]))}")
        else:
            theta = np.arctan2(pt[1], pt[0])
            phi = np.arccos(pt[2]/r)
            print(f"Theta {np.rad2deg(theta)}, Phi: {np.rad2deg(phi)}, Angle: {np.rad2deg(angle_between(point, normal_vectors[key]))}")